In [1]:
# import packages
import asyncio
from playwright.async_api import async_playwright, expect
import random
import time
import datetime
import os
import re
import pandas as pd

## Twitter scraper

This notebook allows you to scrape twitter until either you reach the end of a user's feed or after a certain time is reached. If you think someone has tweeted thousands and thousands of tweets in their lifetime, narrowing down to a set time could be useful. In my experience: ten minutes yielded me nearly 2,000 replies and it took under three minutes to get through 400 posts.

Scrolling to the end was only tested with my twitter, where I have about 100 tweets. Be mindful that this could potentially break in different circumstances.

Right now, this doesn't work for searches but could be easily adapted to do so.

**BEWARE OF PEOPLE / WORDS YOU HAVE MUTED!!! IT WILL AFFECT THE WAY THE SCRAPER WORKS**

In [2]:
# define variables for the scrape
profile_url = "https://x.com/CaoChangqing"
output_file = "CaoChangqing.har"

In [5]:
# copy cookies from chrome application
cookies = pd.read_clipboard(names=["name", "value", "domain", "path", "exp", "size", "httponly", "secure", "samesite", "partition key", "cross site", "priority"])
json_cookies = cookies[["name", "value", "domain", "path"]].to_dict(orient="records")

ParserError: Expected 12 fields in line 6, saw 21. Error could possibly be due to quotes being ignored when a multi-char delimiter is used.

In [5]:
# define function to randomize scroll
async def randomized_scroll(page, range_start, range_end):
    await page.mouse.wheel(0, random.choice(range(range_start,range_end)))
    time.sleep(random.choice(range(1,3))+(random.choice(range(1,1001))/1000))

In [6]:
# let us say we want to try to run this for ten minutes
epoch = time.time() + (10*60)

In [7]:
async def record_har(output=output_file, url=profile_url, cookies=json_cookies, time_stop=0, until_end=True):
    playwright = await async_playwright().start()
    browser = await playwright.chromium.launch(headless=False)
    context = await browser.new_context(record_har_content="embed",
                                        record_har_path=f"data/source/har/{output}",
                                        record_har_mode="minimal",
                                        record_har_url_filter=re.compile(r"https:\/\/x\.com\/i\/api\/graphql\/.*"))
    
    # create context
    page = await context.new_page()

    # add cookies
    await context.add_cookies(cookies)

    # go to the profile url
    await page.goto(url)
    
    try:
        if until_end:
            x = 0
            while x<= 3:
                try:
                    await randomized_scroll(page, 4000, 5000)
                    currentText = await page.get_by_test_id("cellInnerDiv").all_inner_texts()
                    if len(currentText) > 0:
                        if currentText[-1] == '':
                            x += 1
                except:
                    print(str(datetime.datetime.now()))
                    break
        else:
            if time_stop==0:
                print("Will not scroll because time not specified and until end is set to false.")
            while time.time() < time_stop:
                try:
                    await randomized_scroll(page, 4000, 5000)
                except:
                    print(str(datetime.datetime.now()))
                    break
    finally:
        await context.close()
        await browser.close()

In [8]:
await record_har(until_end=True)